In [1]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
import warnings

In [2]:
df=pd.read_csv('C:\\Users\\hp\\OneDrive\\Attachments\\Desktop\\medico\\notebook\\data\\medical.csv')

In [3]:
df.head()

,Unnamed: 0,Name,DateOfBirth,Gender,Symptoms,Causes,Disease,Medicine
0,0,John Doe,15-05-1980,Male,Fever Cough,Viral Infection,Common Cold,Ibuprofen Rest
1,1,Jane Smith,10-08-1992,Female,Headache Fatigue,Stress,Migraine,Sumatriptan
2,2,Michael Lee,20-02-1975,Male,Shortness of breath,Pollution,Asthma,Albuterol Inhaler
3,3,Emily Chen,03-11-1988,Female,Nausea Vomiting,Food Poisoning,Gastroenteritis,Oral RehydrationMedication
4,4,Alex Wong,12-06-2001,Male,Sore Throat,Bacterial Infection,Strep Throat,Penicillin


In [4]:
df.isnull().sum()

Unnamed: 0     0
Name           0
DateOfBirth    0
Gender         0
Symptoms       0
Causes         0
Disease        1
Medicine       0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.dtypes

Unnamed: 0      int64
Name           object
DateOfBirth    object
Gender         object
Symptoms       object
Causes         object
Disease        object
Medicine       object
dtype: object

In [7]:
df.drop(columns=['DateOfBirth'], inplace=True)

In [8]:
df.to_csv("cleaned.csv")

In [9]:
numeric_feature=[feature for feature in df.columns if df[feature].dtype!='O']
categorical_feature=[feature for feature in df.columns if df[feature].dtype=='s']

In [10]:
numeric_feature

['Unnamed: 0']

In [11]:
categorical_feature

[]

In [12]:
num_feature=df.select_dtypes(exclude='O').columns
cat_feature=df.select_dtypes(include='O').columns

In [13]:
X=df.drop(columns=['Medicine'])
y=df['Medicine']

In [14]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:
features=['Gender','Symptoms','Causes','Disease','Medicine']
df_enc=df[features].fillna("")

In [16]:
text_features=(
    
    
    df_enc["Symptoms"]+ " "+ df_enc["Causes"]+ " "+ df_enc["Disease"]+" "+ df_enc["Medicine"]
)

In [17]:
tfidf=TfidfVectorizer(stop_words="english", max_features=5000)

In [18]:
text_encoded=tfidf.fit_transform(text_features)

In [19]:
encoder=OneHotEncoder()
encode=encoder.fit_transform(df_enc[["Gender"]])

In [20]:
from scipy.sparse import hstack

In [21]:
final_features=hstack([text_encoded, encode])

In [22]:
final_feature=final_features.tocsr()

In [23]:
final_feature

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1243 stored elements and shape (160, 185)>

In [24]:
import numpy as np


In [25]:
models={
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(),
    "CatBoost": CatBoostClassifier(verbose=0)
}

In [26]:
model_list=[]

In [27]:
r2_list=[]

In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [29]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_models(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted, average='weighted', zero_division=0)
    recall = recall_score(true, predicted, average='weighted', zero_division=0)
    f1 = f1_score(true, predicted, average='weighted', zero_division=0)
    
    return accuracy, precision, recall, f1

In [ ]:
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(final_feature, y)
    accuracy=model.score(final_feature, y)
    model_list.append(list(models.keys())[i])
    r2_list.append(accuracy)
    predicted=model.predict(final_feature)
    accuracy, precision, recall, f1=evaluate_models(y, predicted)
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")
    print("_" * 30)

Accuracy: 0.8
Precision: 0.7151276154401154
Recall: 0.8
F1 Score: 0.7515029210985094
______________________________
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
______________________________
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
______________________________
Accuracy: 0.9
Precision: 0.8301865842490841
Recall: 0.9
F1 Score: 0.8590983744516354
______________________________
Accuracy: 0.88125
Precision: 0.8022699175824176
Recall: 0.88125
F1 Score: 0.8347733938766547
______________________________


In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=["Model", "r2"]).sort_values(by="r2", ascending=False)

,Model,r2
1,Decision Tree,1.00000
2,Random Forest,1.00000
5,CatBoost,1.00000
3,SVM,0.90000
4,Logistic Regression,0.88125
0,KNN,0.80000
